# 03. Постпроцессинг, метрики и графики

Ноутбук стартует из чистого kernel, загружает `outputs/datasets/*` и `outputs/models/model_artifacts.joblib`, делает inference без `.fit()`, применяет final blend/high-wind post-processing, сохраняет submissions и строит финальные графики.

In [37]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [38]:
# Управление отображением и диагностическими артефактами текущего запуска.
# Финальный submission сохраняется в каталоге текущего запуска.
PLOT_RESEARCH_OUTPUTS = False
PLOT_TWO_STAGE_DIAGNOSTICS = False
SAVE_DIAGNOSTIC_ARTIFACTS = False
SAVE_DIRECT_DEBUG_SUBMISSIONS = False

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [39]:
RANDOM_STATE = 42

In [40]:
# -------------------------
# Физика станции
# -------------------------
INSTALLED_CAPACITY_MW = 90.09
TURBINES_TOTAL = 26

CUT_IN_SPEED = 3.0
RATED_SPEED = 12.0
CUT_OUT_SPEED = 25.0

AIR_DENSITY_REF = 1.225
EPS = 1e-6


In [41]:
# -------------------------
# Пути проекта
# -------------------------
DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
DATA_OUT_DIR = OUT_DIR / "data"
FIGURE_DIR = OUT_DIR / "figures"

for directory in [OUT_DIR, DATA_OUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_CANDIDATES = [
    DATA_DIR / "train_merged.csv",
    DATA_DIR / "train.csv",
]

VALID_CANDIDATES = [
    DATA_DIR / "valid_merged.csv",
    DATA_DIR / "valid.csv",
]

TRAIN_PATH = next((p for p in TRAIN_CANDIDATES if p.exists()), TRAIN_CANDIDATES[0])
VALID_PATH = next((p for p in VALID_CANDIDATES if p.exists()), VALID_CANDIDATES[0])

TURBINE_COORDS_CANDIDATES = [
    Path("map") / "data" / "wind_farm_coords.csv",
    Path("map") / "map_data" / "wind_farm_cords.csv",
    Path("map") / "wind_data" / "wind_farm_cords.csv",
]
TURBINE_COORDS_PATH = next((p for p in TURBINE_COORDS_CANDIDATES if p.exists()), TURBINE_COORDS_CANDIDATES[0])

# Основной файл для отправки.
SUB_PATH = OUT_DIR / "submission_final.csv"

# Диагностические submission-файлы direct ensemble.
DIRECT_SUB_PATH = DATA_OUT_DIR / "direct_full_clipped.csv"
DIRECT_RAW_PATH = DATA_OUT_DIR / "direct_full_raw.csv"

LOG_DIR = DATA_OUT_DIR
RESEARCH_PLOT_DIR = FIGURE_DIR
TS_DIR = DATA_OUT_DIR / "two_stage"


In [42]:
# -------------------------
# Блоки текущего запуска
# -------------------------
FULL_PHYSICS_BLOCK_ENABLED = True
HIGH_WIND_CLIP_ENABLED = True
TWO_STAGE_ENABLED = True

RUN_FINAL_PIPELINE = True
RUN_LOCAL_CHECK = False
PLOT_FINAL_DISTRIBUTIONS = True
RUN_POWER_CURVE_DIAGNOSTIC = True
RUN_HIGH_WIND_CLIP_TUNING = False


In [43]:
# -------------------------
# Веса direct ensemble
# Можно аккуратно крутить веса, но сумма должна быть около 1.
# -------------------------
BLEND_WEIGHTS = {
    "cat_mae_direct": 0.361831,
    "hgb_q545": 0.235409,
    "xgb_residual": 0.177644,
    "hgb_q570": 0.116581,
    "lgb_residual": 0.064663,
    "hgb_q530": 0.043873,
}

# Размер direct ensemble. Если нужно быстрее — уменьши значения или поставь FAST_MODE в отдельных местах.
ENSEMBLE_PARAMS_FULL = {
    "cat_iter": 1200,
    "xgb_estimators": 900,
    "lgb_estimators": 900,
    "hgb_iter": 650,
}

ENSEMBLE_PARAMS_FAST = {
    "cat_iter": 500,
    "xgb_estimators": 450,
    "lgb_estimators": 450,
    "hgb_iter": 350,
}

HGB_QUANTILES = [0.545, 0.570, 0.530]
DIRECT_ENSEMBLE_FAST_MODE = False


In [44]:
# -------------------------
# Геометрия / wake
# -------------------------
N_LAYOUT_CLUSTERS = 4
LAYOUT_KMEANS_N_INIT = 30

WAKE_DIRECTION_STEP_DEG = 5
WAKE_LATERAL_THRESHOLD_M = 260
WAKE_MAX_DOWNWIND_M = 2500
WAKE_DECAY_DOWNWIND_M = 800

BEST_WAKE_FEATURE = "layout_wake_risk_scalar_120m"


In [45]:
# -------------------------
# Физическая декомпозиция скрытых потерь
# -------------------------
FULL_MIN_IDEAL_MW = 3.0
FULL_K_HIDDEN_LOW = -0.25
FULL_K_HIDDEN_HIGH = 0.95
FULL_CHANGEPOINT_WINDOW = 72
FULL_CHANGEPOINT_MIN_DISTANCE = 96
FULL_CHANGEPOINT_THRESHOLD_MAD = 3.5
FULL_MAX_OFF_GRID = TURBINES_TOTAL
FULL_AVAIL_LAMBDA = 0.0015
FULL_UPWIND_WEIGHT_BOOST = 0.35
FULL_OOF_SPLITS = 5
FULL_RANDOM_STATE = RANDOM_STATE


In [46]:
# -------------------------
# Empirical high-wind clip
# Параметры high-wind clip зафиксированы в конфигурации запуска.
# -------------------------
HIGH_WIND_SPEED_COL = "wind_speed_120m"
HIGH_WIND_START_WS = 11.5
HIGH_WIND_TRANSITION = 0.45
HIGH_WIND_CAP_QUANTILE = 0.70
HIGH_WIND_CAP_BIN_WIDTH = 0.50
HIGH_WIND_CAP_MIN_COUNT = 5
HIGH_WIND_CAP_ROLLING_WINDOW = 3
HIGH_WIND_CAP_MARGIN_MW = 1.5
HIGH_WIND_HARD_MAX_CAP = 77.0
HIGH_WIND_CLIP_STRENGTH = 0.85
HIGH_WIND_MIN_PRED = 0.0
HIGH_WIND_MAX_PRED = INSTALLED_CAPACITY_MW

# Cap-кривая как входной сигнал до обучения direct ensemble.
# Target не клипуется: модель получает только оценку high-wind потолка и gate.
HIGH_WIND_CAP_FEATURES_ENABLED = True

# Финальная конфигурация post-processing для submission_final.csv.
FINAL_BENCHMARK_VARIANT = "submission_alpha_0p100_aggressive_q65_cap76"
FINAL_CLIP_CONFIG = {
    "name": "aggressive_q65_cap76",
    "quantile": 0.65,
    "margin_mw": 1.0,
    "hard_max_cap": 76.0,
    "strength": 1.00,
}


In [47]:
# ============================================================
# TWO-STAGE CONFIG: normal behavior + deviation
# ============================================================

TWO_STAGE_ENABLED = True
TWO_STAGE_FAST_MODE = False

# Количество time-series split для OOF normal/deviation
TWO_STAGE_N_SPLITS = 5

# Финальная смесь:
# final = (1 - alpha) * direct_full + alpha * two_stage
# Коэффициент смешивания задаёт долю two-stage поправки в финальном прогнозе.
FINAL_TWO_STAGE_ALPHA = 0.10

# ============================================================
# Residual / deviation target clipping
# ============================================================
# Квантильная обрезка оставляет residual-модели редкие режимы станции.
TWO_STAGE_RESIDUAL_Q_LOW = 0.005
TWO_STAGE_RESIDUAL_Q_HIGH = 0.995
TWO_STAGE_RESIDUAL_CLIP_MULT = 1.25

# ============================================================
# Deviation prediction safety
# ============================================================
# Не зажимаем deviation: residual-ветка должна иметь право на крупную поправку.
# Значение 999 фактически отключает дополнительный absolute clip,
# остаётся только квантильная обрезка target/prediction.
TWO_STAGE_DEVIATION_SHRINK = 1.00
TWO_STAGE_DEVIATION_ABS_CLIP_MW = 999.0

# ============================================================
# Bad OOF head handling
# ============================================================
# Ранний OOF-участок участвует в обучении residual-модели.
TWO_STAGE_DROP_BAD_OOF_HEAD = False
TWO_STAGE_BAD_HEAD_FRAC = 0.00

# ============================================================
# Physics gate
# ============================================================
# Gate равен 1.0: residual-поправка проходит без дополнительного приглушения.
TWO_STAGE_USE_PHYSICS_GATE = False
TWO_STAGE_GATE_MIN = 1.00
TWO_STAGE_GATE_MAX = 1.00

# Какие признаки убрать из normal-ветки, чтобы normal-модель была именно про нормальную генерацию,
# а не про скрытые потери/аномалии.
TWO_STAGE_DEVIATION_KEYWORDS = [
    "full_k_hidden",
    "full_hidden_loss",
    "full_p_reconstructed",
    "full_recon_minus",
    "full_k_meteo",
    "full_k_perf",
    "full_k_aging",
    "phi_ice",
    "phi_turbulence",
    "phi_yaw",
]

TS_DIR = OUT_DIR / "ts"
TS_PLOTS_DIR = TS_DIR / "plots"

display(Markdown(
    "### Контрольный снимок конфигурации\n\n"
    "Пути, физические блоки, high-wind clip и two-stage параметры текущего запуска."
))

config_snapshot = pd.DataFrame([
    {"group": "paths", "parameter": "TRAIN_PATH", "value": str(TRAIN_PATH)},
    {"group": "paths", "parameter": "VALID_PATH", "value": str(VALID_PATH)},
    {"group": "paths", "parameter": "TURBINE_COORDS_PATH", "value": str(TURBINE_COORDS_PATH)},
    {"group": "paths", "parameter": "OUT_DIR", "value": str(OUT_DIR)},
    {"group": "paths", "parameter": "SUB_PATH", "value": str(SUB_PATH)},
    {"group": "paths", "parameter": "SAVE_DIAGNOSTIC_ARTIFACTS", "value": SAVE_DIAGNOSTIC_ARTIFACTS},
    {"group": "paths", "parameter": "SAVE_DIRECT_DEBUG_SUBMISSIONS", "value": SAVE_DIRECT_DEBUG_SUBMISSIONS},
    {"group": "direct", "parameter": "BLEND_WEIGHTS_SUM", "value": sum(BLEND_WEIGHTS.values())},
    {"group": "direct", "parameter": "DIRECT_ENSEMBLE_FAST_MODE", "value": DIRECT_ENSEMBLE_FAST_MODE},
    {"group": "physics", "parameter": "FULL_PHYSICS_BLOCK_ENABLED", "value": FULL_PHYSICS_BLOCK_ENABLED},
    {"group": "clip", "parameter": "HIGH_WIND_CLIP_ENABLED", "value": HIGH_WIND_CLIP_ENABLED},
    {"group": "clip", "parameter": "HIGH_WIND_SPEED_COL", "value": HIGH_WIND_SPEED_COL},
    {"group": "clip", "parameter": "HIGH_WIND_CAP_FEATURES_ENABLED", "value": HIGH_WIND_CAP_FEATURES_ENABLED},
    {"group": "two_stage", "parameter": "TWO_STAGE_ENABLED", "value": TWO_STAGE_ENABLED},
    {"group": "two_stage", "parameter": "FINAL_TWO_STAGE_ALPHA", "value": FINAL_TWO_STAGE_ALPHA},
    {"group": "final", "parameter": "FINAL_BENCHMARK_VARIANT", "value": FINAL_BENCHMARK_VARIANT},
    {"group": "final", "parameter": "FINAL_CLIP_CONFIG", "value": str(FINAL_CLIP_CONFIG)},
    {"group": "two_stage", "parameter": "TWO_STAGE_N_SPLITS", "value": TWO_STAGE_N_SPLITS},
    {"group": "two_stage", "parameter": "TWO_STAGE_USE_PHYSICS_GATE", "value": TWO_STAGE_USE_PHYSICS_GATE},
    {"group": "two_stage", "parameter": "TWO_STAGE_DROP_BAD_OOF_HEAD", "value": TWO_STAGE_DROP_BAD_OOF_HEAD},
    {"group": "two_stage", "parameter": "TWO_STAGE_RESIDUAL_Q", "value": f"{TWO_STAGE_RESIDUAL_Q_LOW} .. {TWO_STAGE_RESIDUAL_Q_HIGH}"},
])
display(config_snapshot)

### Контрольный снимок конфигурации

Пути, физические блоки, high-wind clip и two-stage параметры текущего запуска.

,group,parameter,value
0,paths,TRAIN_PATH,data\train_merged.csv
1,paths,VALID_PATH,data\valid_merged.csv
2,paths,TURBINE_COORDS_PATH,map\data\wind_farm_coords.csv
3,paths,OUT_DIR,outputs
4,paths,SUB_PATH,outputs\submission_final.csv
5,paths,SAVE_DIAGNOSTIC_ARTIFACTS,False
6,paths,SAVE_DIRECT_DEBUG_SUBMISSIONS,False
7,direct,BLEND_WEIGHTS_SUM,1.000001
8,direct,DIRECT_ENSEMBLE_FAST_MODE,False
9,physics,FULL_PHYSICS_BLOCK_ENABLED,True


In [48]:
TARGET_CANDIDATES = [
    "Выработка. Результирующий расчет",
    "target",
    "Выработка",
]

DATETIME_CANDIDATES = [
    "METEOFORECASTHOUR_OPENM_Datetime",
    "Datetime",
    "datetime",
    "date",
    "time",
]

REPAIR_CANDIDATES = [
    "Кол-во_ВЭУ_в_ремонте",
    "turbines_in_repair",
    "repair",
]

In [49]:
# Для исследовательского вида оставляем локальные таблицы и ключевые графики в ноутбуке.
PLOT_RESEARCH_OUTPUTS = True
PLOT_FINAL_DISTRIBUTIONS = False
RUN_POWER_CURVE_DIAGNOSTIC = False
RUN_LOCAL_CHECK = False
SAVE_DIAGNOSTIC_ARTIFACTS = True
SAVE_DIRECT_DEBUG_SUBMISSIONS = True

DATASET_OUT_DIR = OUT_DIR / "datasets"
MODEL_OUT_DIR = OUT_DIR / "models"
for directory in [DATASET_OUT_DIR, MODEL_OUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


## 1. Загрузка готовых features и весов

In [50]:
import joblib
import ves_pipeline_core as vpc

feature_contract = vpc.validate_feature_contract()
model_artifacts = joblib.load(vpc.MODEL_ARTIFACTS_PATH)
valid_features = pd.read_csv(vpc.VALID_FEATURES_PATH)
test_features = pd.read_csv(vpc.TEST_FEATURES_PATH)
test_actual = pd.read_csv(vpc.TEST_ACTUAL_PATH)
display(feature_contract)
display(pd.DataFrame([
    {"artifact": "model_artifacts", "path": str(vpc.MODEL_ARTIFACTS_PATH), "exists": vpc.MODEL_ARTIFACTS_PATH.exists()},
    {"artifact": "valid_features", "path": str(vpc.VALID_FEATURES_PATH), "rows": len(valid_features)},
    {"artifact": "test_features", "path": str(vpc.TEST_FEATURES_PATH), "rows": len(test_features)},
]))


,dataset,feature_rows,raw_rows,row_count_ok,missing_model_features
0,train,32434,32434,True,0
1,valid,2126,2126,True,0
2,test,1152,1152,True,0


,artifact,path,exists,rows
0,model_artifacts,outputs\models\model_artifacts.joblib,True,NaN
1,valid_features,outputs\datasets\valid_features.csv,NaN,2126.0
2,test_features,outputs\datasets\test_features.csv,NaN,1152.0


## 2. Inference без переобучения

In [51]:
# Sequential notebook step; kept inline instead of a one-shot wrapper.
inference_env = vpc._prepare_training_env()
model_artifacts = joblib.load(vpc.MODEL_ARTIFACTS_PATH)
model_features = model_artifacts["model_features"]
capacity = model_artifacts["constants"]["INSTALLED_CAPACITY_MW"]

valid_features = pd.read_csv(vpc.VALID_FEATURES_PATH)
test_features = pd.read_csv(vpc.TEST_FEATURES_PATH)
for frame in [valid_features, test_features]:
    for col in model_features:
        if col not in frame.columns:
            frame[col] = np.nan
    if "datetime" in frame.columns:
        frame["datetime"] = pd.to_datetime(frame["datetime"], errors="coerce")

valid_pred, valid_diag = vpc._final_blend(inference_env, model_artifacts, valid_features)
test_pred, test_diag = vpc._final_blend(inference_env, model_artifacts, test_features)
valid_submission = vpc._save_submission(valid_features, valid_pred, vpc.FINAL_VALID_SUBMISSION_PATH)
test_submission = vpc._save_submission(test_features, test_pred, vpc.TEST_SUBMISSION_PATH)

test_actual = pd.read_csv(vpc.TEST_ACTUAL_PATH)
test_compare = test_actual.sort_values("row_id").reset_index(drop=True).copy()
test_compare["prediction_mw"] = test_submission["target"].to_numpy(dtype=float)
if "actual_mw" not in test_compare.columns:
    test_compare["actual_mw"] = np.nan
test_compare["error_mw"] = test_compare["prediction_mw"] - test_compare["actual_mw"]
test_compare["abs_error_mw"] = test_compare["error_mw"].abs()
test_metrics = vpc._metrics(test_compare)

test_compare.to_csv(vpc.TEST_COMPARE_PATH, index=False)
test_metrics.to_csv(vpc.TEST_METRICS_PATH, index=False)
valid_diag.to_csv(vpc.OUTPUT_DIR / "valid_prediction_diagnostics.csv", index=False)
test_diag.to_csv(vpc.TEST_OUTPUT_DIR / "prediction_diagnostics.csv", index=False)

if "datetime" in test_compare.columns:
    dt = pd.to_datetime(test_compare["datetime"], errors="coerce")
    day_mask = dt.dt.date.astype(str).eq("2026-05-18")
    if day_mask.any():
        day_submission = pd.DataFrame({"target": test_compare.loc[day_mask, "prediction_mw"].to_numpy(dtype=float)})
    else:
        day_submission = test_submission.tail(24).reset_index(drop=True)
else:
    day_submission = test_submission.tail(24).reset_index(drop=True)
day_submission.to_csv(vpc.TEST_DAY_SUBMISSION_PATH, index=False)

vpc._plot_test_diagnostics(test_compare, test_metrics, capacity)
TEST_FIGURE_DIR = vpc.TEST_FIGURE_DIR

display(Markdown("### Metrics on test rows with known target"))
display(test_metrics)


### Контрольный снимок конфигурации

Пути, физические блоки, high-wind clip и two-stage параметры текущего запуска.

,group,parameter,value
0,paths,TRAIN_PATH,data\train_merged.csv
1,paths,VALID_PATH,data\valid_merged.csv
2,paths,TURBINE_COORDS_PATH,map\data\wind_farm_coords.csv
3,paths,OUT_DIR,outputs
4,paths,SUB_PATH,outputs\submission_final.csv
5,paths,SAVE_DIAGNOSTIC_ARTIFACTS,False
6,paths,SAVE_DIRECT_DEBUG_SUBMISSIONS,False
7,direct,BLEND_WEIGHTS_SUM,1.000001
8,direct,DIRECT_ENSEMBLE_FAST_MODE,False
9,physics,FULL_PHYSICS_BLOCK_ENABLED,True


N_LAYOUT_CLUSTERS: 4
WAKE_DIRECTION_STEP_DEG: 5
BEST_WAKE_FEATURE: layout_wake_risk_scalar_120m
ГЕОМЕТРИЯ ВЭС
Турбин: 26
Среднее ближайшее расстояние: 343.0 м
Медианное ближайшее расстояние: 353.6 м
Главная ось ВЭС: 9.30° / 189.30°
Доля объяснённой дисперсии PCA: [0.69031765 0.30968235]
BLEND_WEIGHTS sum: 1.000001
ENSEMBLE_PARAMS_FULL: {'cat_iter': 1200, 'xgb_estimators': 900, 'lgb_estimators': 900, 'hgb_iter': 650}
HGB_QUANTILES: [0.545, 0.57, 0.53]
FULL_PHYSICS_BLOCK_ENABLED: True
FULL_OOF_SPLITS: 5
FULL_CHANGEPOINT_WINDOW: 72
FULL_AVAIL_LAMBDA: 0.0015
TWO_STAGE_ENABLED: True
TWO_STAGE_FAST_MODE: False
TWO_STAGE_N_SPLITS: 5
FINAL_TWO_STAGE_ALPHA: 0.1
TS_DIR: outputs\ts


### Metrics on test rows with known target

,rows_total,rows_with_actual,rows_without_actual,mae_mw,rmse_mw,bias_mw,median_abs_error_mw,p90_abs_error_mw,max_abs_error_mw,corr_actual_prediction,actual_mean_mw,prediction_mean_mw
0,1152,1128,24,7.066174,10.497052,0.61807,4.213574,16.762823,51.811081,0.896165,23.233752,23.851821


## 3. Submission integrity

In [52]:
valid_submission = pd.read_csv(vpc.FINAL_VALID_SUBMISSION_PATH)
test_submission = pd.read_csv(vpc.TEST_SUBMISSION_PATH)
day_submission = pd.read_csv(vpc.TEST_DAY_SUBMISSION_PATH)
display(pd.DataFrame([
    {"submission": "valid", "path": str(vpc.FINAL_VALID_SUBMISSION_PATH), "rows": len(valid_submission), "columns": list(valid_submission.columns), "min": valid_submission["target"].min(), "max": valid_submission["target"].max(), "nan": int(valid_submission["target"].isna().sum())},
    {"submission": "test", "path": str(vpc.TEST_SUBMISSION_PATH), "rows": len(test_submission), "columns": list(test_submission.columns), "min": test_submission["target"].min(), "max": test_submission["target"].max(), "nan": int(test_submission["target"].isna().sum())},
    {"submission": "test_2026_05_18", "path": str(vpc.TEST_DAY_SUBMISSION_PATH), "rows": len(day_submission), "columns": list(day_submission.columns), "min": day_submission["target"].min(), "max": day_submission["target"].max(), "nan": int(day_submission["target"].isna().sum())},
]))


,submission,path,rows,columns,min,max,nan
0,valid,outputs\submission_final.csv,2126,[target],0.000000,76.902971,0
1,test,outputs\test\submission_final.csv,1152,[target],0.184094,76.043774,0
2,test_2026_05_18,outputs\test\submission_2026-05-18.csv,23,[target],0.184094,3.881468,0


## 4. Таблицы ошибок и финальные графики

In [53]:
# ============================================================
# TEST DATASET: visual diagnostics and result interpretation
# ============================================================

from IPython.display import Markdown, display

_required = ["test_compare", "test_metrics", "TEST_FIGURE_DIR"]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(
        "Сначала выполни test-eval ячейку. Не хватает объектов: "
        + ", ".join(_missing)
    )

TEST_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

eval_df = test_compare[
    test_compare["actual_mw"].notna() &
    test_compare["prediction_mw"].notna()
].copy()

if len(eval_df) == 0:
    raise RuntimeError("Нет строк с фактической выработкой для оценки.")

eval_df["datetime"] = pd.to_datetime(eval_df["datetime"], errors="coerce")
eval_df["error_mw"] = eval_df["prediction_mw"] - eval_df["actual_mw"]
eval_df["abs_error_mw"] = eval_df["error_mw"].abs()

metrics = test_metrics.iloc[0].to_dict()

def save_show(name):
    path = TEST_FIGURE_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

display(Markdown("## Test dataset diagnostics"))

display(pd.DataFrame([{
    "rows_total": int(metrics.get("rows_total", len(test_compare))),
    "rows_with_actual": int(metrics.get("rows_with_actual", len(eval_df))),
    "rows_without_actual": int(metrics.get("rows_without_actual", test_compare["actual_mw"].isna().sum())),
    "mae_mw": metrics.get("mae_mw", np.nan),
    "rmse_mw": metrics.get("rmse_mw", np.nan),
    "bias_mw": metrics.get("bias_mw", np.nan),
    "median_abs_error_mw": metrics.get("median_abs_error_mw", np.nan),
    "p90_abs_error_mw": metrics.get("p90_abs_error_mw", np.nan),
    "max_abs_error_mw": metrics.get("max_abs_error_mw", np.nan),
    "corr_actual_prediction": metrics.get("corr_actual_prediction", np.nan),
    "actual_mean_mw": metrics.get("actual_mean_mw", np.nan),
    "prediction_mean_mw": metrics.get("prediction_mean_mw", np.nan),
}]))

# ------------------------------------------------------------
# 1. Прогноз против факта во времени
# ------------------------------------------------------------

plot_df = eval_df.sort_values("datetime").copy()

plt.figure(figsize=(16, 5))
plt.plot(plot_df["datetime"], plot_df["actual_mw"], linewidth=1.8, label="Факт")
plt.plot(plot_df["datetime"], plot_df["prediction_mw"], linewidth=1.4, alpha=0.85, label="Прогноз")
plt.title("Test dataset: прогноз против фактической выработки")
plt.xlabel("Дата")
plt.ylabel("Выработка, МВт")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
save_show("test_01_prediction_vs_actual_time.png")

# ------------------------------------------------------------
# 2. Ошибка во времени
# ------------------------------------------------------------

plt.figure(figsize=(16, 4.5))
plt.plot(plot_df["datetime"], plot_df["error_mw"], linewidth=1.2)
plt.axhline(0, linestyle="--", linewidth=1.2, color="black")
plt.title("Test dataset: ошибка прогноза во времени")
plt.xlabel("Дата")
plt.ylabel("prediction - actual, МВт")
plt.grid(alpha=0.25)
plt.tight_layout()
save_show("test_02_error_time.png")

# ------------------------------------------------------------
# 3. Scatter: факт vs прогноз
# ------------------------------------------------------------

lims = [
    min(eval_df["actual_mw"].min(), eval_df["prediction_mw"].min(), 0),
    max(eval_df["actual_mw"].max(), eval_df["prediction_mw"].max(), INSTALLED_CAPACITY_MW),
]

plt.figure(figsize=(7, 7))
plt.scatter(
    eval_df["actual_mw"],
    eval_df["prediction_mw"],
    s=18,
    alpha=0.55,
)
plt.plot(lims, lims, linestyle="--", linewidth=1.5, color="black", label="Идеально")
plt.xlim(lims)
plt.ylim(lims)
plt.title("Test dataset: факт vs прогноз")
plt.xlabel("Факт, МВт")
plt.ylabel("Прогноз, МВт")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
save_show("test_03_actual_vs_prediction_scatter.png")

# ------------------------------------------------------------
# 4. Распределение абсолютной ошибки
# ------------------------------------------------------------

plt.figure(figsize=(12, 4.5))
plt.hist(eval_df["abs_error_mw"], bins=45, alpha=0.85)
plt.axvline(metrics.get("mae_mw", eval_df["abs_error_mw"].mean()), linestyle="--", linewidth=1.5, color="black", label="MAE")
plt.axvline(metrics.get("median_abs_error_mw", eval_df["abs_error_mw"].median()), linestyle=":", linewidth=2.0, color="black", label="Median AE")
plt.title("Test dataset: распределение абсолютной ошибки")
plt.xlabel("|prediction - actual|, МВт")
plt.ylabel("Количество")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
save_show("test_04_abs_error_distribution.png")

# ------------------------------------------------------------
# 5. Ошибка по бинам фактической мощности
# ------------------------------------------------------------

power_bins = [0, 1, 5, 10, 20, 40, 60, 80, INSTALLED_CAPACITY_MW + 1]
power_labels = ["0-1", "1-5", "5-10", "10-20", "20-40", "40-60", "60-80", "80+"]

eval_df["actual_power_bin"] = pd.cut(
    eval_df["actual_mw"].clip(0, INSTALLED_CAPACITY_MW),
    bins=power_bins,
    labels=power_labels,
    include_lowest=True,
)

power_bin_report = (
    eval_df
    .groupby("actual_power_bin", observed=True)
    .agg(
        rows=("abs_error_mw", "size"),
        mae_mw=("abs_error_mw", "mean"),
        bias_mw=("error_mw", "mean"),
        actual_mean_mw=("actual_mw", "mean"),
        pred_mean_mw=("prediction_mw", "mean"),
    )
    .reset_index()
)

display(Markdown("### Ошибка по бинам фактической мощности"))
display(power_bin_report)

plt.figure(figsize=(12, 4.5))
plt.bar(power_bin_report["actual_power_bin"].astype(str), power_bin_report["mae_mw"])
plt.title("Test dataset: MAE по бинам фактической мощности")
plt.xlabel("Фактическая мощность, МВт")
plt.ylabel("MAE, МВт")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
save_show("test_05_mae_by_actual_power_bin.png")

# ------------------------------------------------------------
# 6. Ошибка по скорости ветра 120м
# ------------------------------------------------------------

if "wind_speed_120m" in eval_df.columns:
    wind_bins = [0, 3, 5, 7, 9, 11, 13, 15, 30]
    wind_labels = ["0-3", "3-5", "5-7", "7-9", "9-11", "11-13", "13-15", "15+"]

    eval_df["wind_120m_bin"] = pd.cut(
        eval_df["wind_speed_120m"],
        bins=wind_bins,
        labels=wind_labels,
        include_lowest=True,
    )

    wind_bin_report = (
        eval_df
        .groupby("wind_120m_bin", observed=True)
        .agg(
            rows=("abs_error_mw", "size"),
            mae_mw=("abs_error_mw", "mean"),
            bias_mw=("error_mw", "mean"),
            actual_mean_mw=("actual_mw", "mean"),
            pred_mean_mw=("prediction_mw", "mean"),
        )
        .reset_index()
    )

    display(Markdown("### Ошибка по бинам скорости ветра 120м"))
    display(wind_bin_report)

    plt.figure(figsize=(12, 4.5))
    plt.bar(wind_bin_report["wind_120m_bin"].astype(str), wind_bin_report["mae_mw"])
    plt.title("Test dataset: MAE по скорости ветра 120м")
    plt.xlabel("wind_speed_120m, м/с")
    plt.ylabel("MAE, МВт")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    save_show("test_06_mae_by_wind_speed_120m_bin.png")

# ------------------------------------------------------------
# 7. Топ худших ошибок
# ------------------------------------------------------------

worst_errors = (
    eval_df
    .sort_values("abs_error_mw", ascending=False)
    .head(20)
    .copy()
)

display(Markdown("### Топ-20 худших ошибок"))
display(worst_errors)

# ------------------------------------------------------------
# 8. Короткая текстовая оценка
# ------------------------------------------------------------

mae = float(metrics.get("mae_mw", eval_df["abs_error_mw"].mean()))
rmse = float(metrics.get("rmse_mw", np.sqrt(np.mean(eval_df["error_mw"] ** 2))))
bias = float(metrics.get("bias_mw", eval_df["error_mw"].mean()))
median_ae = float(metrics.get("median_abs_error_mw", eval_df["abs_error_mw"].median()))
p90_ae = float(metrics.get("p90_abs_error_mw", eval_df["abs_error_mw"].quantile(0.90)))
cap_share = mae / INSTALLED_CAPACITY_MW * 100

if abs(bias) < 1.0:
    bias_text = "систематический сдвиг маленький"
elif bias > 0:
    bias_text = "модель в среднем завышает прогноз"
else:
    bias_text = "модель в среднем занижает прогноз"

if rmse / mae > 1.45:
    tail_text = "есть заметный хвост крупных ошибок"
else:
    tail_text = "крупные ошибки не доминируют"

display(Markdown(f"""
### Интерпретация результата

- **MAE = {mae:.3f} МВт** — средняя ошибка около **{cap_share:.2f}%** от установленной мощности {INSTALLED_CAPACITY_MW:.2f} МВт.
- **Median AE = {median_ae:.3f} МВт** — типичная ошибка заметно ниже MAE, значит большая часть часов предсказывается нормально.
- **RMSE = {rmse:.3f} МВт**, **P90 AE = {p90_ae:.3f} МВт** — {tail_text}; основные проблемы сидят в отдельных сложных режимах.
- **Bias = {bias:.3f} МВт** — {bias_text}.
"""))

## Test dataset diagnostics

,rows_total,rows_with_actual,rows_without_actual,mae_mw,rmse_mw,bias_mw,median_abs_error_mw,p90_abs_error_mw,max_abs_error_mw,corr_actual_prediction,actual_mean_mw,prediction_mean_mw
0,1152,1128,24,7.066174,10.497052,0.61807,4.213574,16.762823,51.811081,0.896165,23.233752,23.851821


Saved: outputs\test\figures\test_01_prediction_vs_actual_time.png
Saved: outputs\test\figures\test_02_error_time.png
Saved: outputs\test\figures\test_03_actual_vs_prediction_scatter.png
Saved: outputs\test\figures\test_04_abs_error_distribution.png


### Ошибка по бинам фактической мощности

,actual_power_bin,rows,mae_mw,bias_mw,actual_mean_mw,pred_mean_mw
0,0-1,187,4.067222,4.067222,0.181701,4.248923
1,1-5,145,4.651843,4.139746,2.801076,6.940822
2,5-10,123,5.923083,3.309933,7.415935,10.725868
3,10-20,188,7.477155,2.699579,14.924798,17.624377
4,20-40,236,8.714109,-1.647347,29.056174,27.408827
5,40-60,115,12.796691,-6.502505,48.832000,42.329495
6,60-80,129,6.543720,-3.199295,71.111318,67.912023
7,80+,5,5.804602,-5.804602,80.643800,74.839198


...

,wind_120m_bin,rows,mae_mw,bias_mw,actual_mean_mw,pred_mean_mw
0,0-3,195,2.505794,0.985145,1.643836,2.628981
1,3-5,264,4.903525,-0.798694,7.753784,6.955090
2,5-7,292,8.537083,-1.272053,20.522158,19.250105
3,7-9,199,10.734161,2.598529,34.460477,37.059007
4,9-11,88,12.436378,4.154202,52.565886,56.720088
5,11-13,49,6.653600,3.063291,67.419796,70.483087
6,13-15,37,3.237347,1.079645,74.900000,75.979645
7,15+,4,4.587006,3.662170,72.338000,76.000170


Saved: outputs\test\figures\test_06_mae_by_wind_speed_120m_bin.png


### Топ-20 худших ошибок

,row_id,datetime,actual_mw,wind_speed_120m,wind_speed_80m,wind_direction_120m,Кол-во_ВЭУ_в_ремонте,prediction_mw,error_mw,abs_error_mw,actual_power_bin,wind_120m_bin
817,817,2026-04-14 22:00:00,6.309,10.60,8.41,0.244,3,58.120081,51.811081,51.811081,5-10,9-11
816,816,2026-04-14 23:00:00,1.234,10.07,8.06,0.249,3,50.828667,49.594667,49.594667,1-5,9-11
813,813,2026-04-15 02:00:00,0.001,8.98,7.65,0.262,3,47.578782,47.577782,47.577782,0-1,7-9
771,771,2026-04-16 20:00:00,52.110,4.54,3.93,0.278,3,6.371974,-45.738026,45.738026,40-60,3-5
812,812,2026-04-15 03:00:00,0.001,9.21,7.60,0.272,3,44.949326,44.948326,44.948326,0-1,9-11
814,814,2026-04-15 01:00:00,0.852,8.56,6.96,0.256,3,45.386291,44.534291,44.534291,0-1,7-9
815,815,2026-04-15 00:00:00,4.189,9.00,7.02,0.246,3,46.758855,42.569855,42.569855,1-5,7-9
983,983,2026-04-08 00:00:00,12.564,9.06,7.56,0.211,3,54.147387,41.583387,41.583387,10-20,9-11
985,985,2026-04-07 22:00:00,16.775,9.26,7.98,0.213,3,57.409307,40.634307,40.634307,10-20,9-11
934,934,2026-04-10 01:00:00,64.246,6.16,4.83,0.228,3,23.924577,-40.321423,40.321423,60-80,5-7



### Интерпретация результата

- **MAE = 7.066 МВт** — средняя ошибка около **7.84%** от установленной мощности 90.09 МВт.
- **Median AE = 4.214 МВт** — типичная ошибка заметно ниже MAE, значит большая часть часов предсказывается нормально.
- **RMSE = 10.497 МВт**, **P90 AE = 16.763 МВт** — есть заметный хвост крупных ошибок; основные проблемы сидят в отдельных сложных режимах.
- **Bias = 0.618 МВт** — систематический сдвиг маленький.


In [54]:
display(Markdown("### Самые крупные ошибки на test"))
eval_rows = test_compare[test_compare["actual_mw"].notna()].copy()
display(eval_rows.sort_values("abs_error_mw", ascending=False).head(15))

display(Markdown("### Финальные файлы"))
display(pd.DataFrame([
    {"artifact": "valid_submission", "path": str(vpc.FINAL_VALID_SUBMISSION_PATH), "rows": len(valid_submission)},
    {"artifact": "test_submission", "path": str(vpc.TEST_SUBMISSION_PATH), "rows": len(test_submission)},
    {"artifact": "test_day_submission", "path": str(vpc.TEST_DAY_SUBMISSION_PATH), "rows": len(day_submission)},
    {"artifact": "test_compare", "path": str(vpc.TEST_COMPARE_PATH), "rows": len(test_compare)},
    {"artifact": "test_metrics", "path": str(vpc.TEST_METRICS_PATH), "rows": len(test_metrics)},
    {"artifact": "test_figures_dir", "path": str(vpc.TEST_FIGURE_DIR), "rows": len(list(vpc.TEST_FIGURE_DIR.glob("*.png")))},
]))


### Самые крупные ошибки на test

,row_id,datetime,actual_mw,wind_speed_120m,wind_speed_80m,wind_direction_120m,Кол-во_ВЭУ_в_ремонте,prediction_mw,error_mw,abs_error_mw
817,817,2026-04-14 22:00:00.000,6.309,10.60,8.41,0.244,3,58.120081,51.811081,51.811081
816,816,2026-04-14 23:00:00.000,1.234,10.07,8.06,0.249,3,50.828667,49.594667,49.594667
813,813,2026-04-15 02:00:00.000,0.001,8.98,7.65,0.262,3,47.578782,47.577782,47.577782
771,771,2026-04-16 20:00:00.000,52.110,4.54,3.93,0.278,3,6.371974,-45.738026,45.738026
812,812,2026-04-15 03:00:00.000,0.001,9.21,7.60,0.272,3,44.949326,44.948326,44.948326
814,814,2026-04-15 01:00:00.000,0.852,8.56,6.96,0.256,3,45.386291,44.534291,44.534291
815,815,2026-04-15 00:00:00.000,4.189,9.00,7.02,0.246,3,46.758855,42.569855,42.569855
983,983,2026-04-08 00:00:00.000,12.564,9.06,7.56,0.211,3,54.147387,41.583387,41.583387
985,985,2026-04-07 22:00:00.000,16.775,9.26,7.98,0.213,3,57.409307,40.634307,40.634307
934,934,2026-04-10 01:00:00.000,64.246,6.16,4.83,0.228,3,23.924577,-40.321423,40.321423


### Финальные файлы

,artifact,path,rows
0,valid_submission,outputs\submission_final.csv,2126
1,test_submission,outputs\test\submission_final.csv,1152
2,test_day_submission,outputs\test\submission_2026-05-18.csv,23
3,test_compare,outputs\test\test_prediction_vs_actual.csv,1152
4,test_metrics,outputs\test\test_prediction_metrics.csv,1
5,test_figures_dir,outputs\test\figures,6
